# Feature Stores & Data Infrastructure — Azure ML Feature Store

**Chapter 8: AI Infrastructure (Azure Supplement)**

This notebook demonstrates production-grade feature store deployment using **Azure ML Feature Store** with Redis as the online store.

**Production Architecture:**
- **Offline store**: Azure Data Lake Storage (ADLS Gen2) with Parquet files
- **Online store**: Azure Cache for Redis (Enterprise tier, 99.99% SLA)
- **Feature serving**: Azure Functions (serverless, auto-scaling)
- **Monitoring**: Application Insights (latency, drift, staleness)

**Tools:**
- **Azure ML SDK**: Feature store management
- **Azure Cache for Redis**: Managed Redis (6GB-120GB)
- **Azure Functions**: Serverless feature serving
- **Application Insights**: Telemetry and monitoring

**Cost targets:**
- Redis: $120-$600/month (depending on tier)
- Azure Functions: $0.20/million requests
- ADLS: $0.02/GB/month

---

## Cell 1: Azure Credentials + ML Workspace Setup

**Required Azure resources:**
1. Azure ML Workspace
2. Azure Cache for Redis (Standard/Premium tier)
3. Azure Data Lake Storage Gen2
4. Service Principal with Contributor role

**Security note:** This cell contains credential placeholders. In production:
- Use Azure Key Vault for secrets
- Use Managed Identity for Azure resources
- Never commit credentials to Git

In [ ]:
# Install Azure ML SDK
!pip install azure-ai-ml azure-identity azureml-fsspec redis pandas pyarrow matplotlib -q

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, ClientSecretCredential
import os

# ==========================================
# CREDENTIALS (REPLACE WITH YOUR VALUES)
# ==========================================
# Option 1: Use DefaultAzureCredential (recommended for local dev with Azure CLI login)
credential = DefaultAzureCredential()

# Option 2: Service Principal (for CI/CD pipelines)
# credential = ClientSecretCredential(
#     tenant_id="YOUR_TENANT_ID",
#     client_id="YOUR_CLIENT_ID",
#     client_secret="YOUR_CLIENT_SECRET"
# )

# Azure ML Workspace configuration
SUBSCRIPTION_ID = "YOUR_SUBSCRIPTION_ID"  # ← REPLACE
RESOURCE_GROUP = "rg-inferencebase-prod"  # ← REPLACE
WORKSPACE_NAME = "mlw-inferencebase-prod"  # ← REPLACE

# Azure Cache for Redis configuration
REDIS_HOST = "inferencebase-redis.redis.cache.windows.net"  # ← REPLACE
REDIS_PORT = 6380  # SSL port
REDIS_PASSWORD = "YOUR_REDIS_PRIMARY_KEY"  # ← REPLACE (from Azure Portal)

# Azure Data Lake Storage
ADLS_ACCOUNT_NAME = "stinferencebase"  # ← REPLACE
ADLS_CONTAINER = "features"  # Container for feature data

# ==========================================

# Initialize ML Client
try:
    ml_client = MLClient(
        credential=credential,
        subscription_id=SUBSCRIPTION_ID,
        resource_group_name=RESOURCE_GROUP,
        workspace_name=WORKSPACE_NAME
    )
    workspace = ml_client.workspaces.get(WORKSPACE_NAME)
    print(f"✓ Connected to Azure ML Workspace: {workspace.name}")
    print(f"  Location: {workspace.location}")
    print(f"  Subscription: {SUBSCRIPTION_ID[:8]}...")
except Exception as e:
    print(f"❌ Error connecting to Azure ML: {e}")
    print(f"\n⚠️  Make sure you have:")
    print(f"  1. Logged in via: az login")
    print(f"  2. Set correct subscription: az account set --subscription {SUBSCRIPTION_ID}")
    print(f"  3. Created ML workspace: {WORKSPACE_NAME}")

# Test Redis connection
import redis
try:
    r = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        password=REDIS_PASSWORD,
        ssl=True,
        ssl_cert_reqs=None
    )
    r.ping()
    print(f"\n✓ Connected to Azure Cache for Redis: {REDIS_HOST}")
except Exception as e:
    print(f"\n❌ Error connecting to Redis: {e}")
    print(f"  Check: REDIS_HOST, REDIS_PASSWORD in Azure Portal")

## Cell 2: Upload Training Data to Azure Data Lake

Upload user activity data to ADLS Gen2 (offline store).

**Storage layout:**
```
features/
├── user_uploads/
│   └── year=2024/month=04/day=26/
│       └── user_uploads.parquet
├── user_activity_features/
└── user_doc_affinity/
```

**Cost:** $0.02/GB/month (ADLS Standard tier)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from azure.storage.filedatalake import DataLakeServiceClient

# Generate synthetic user upload data (same as main notebook)
np.random.seed(42)
n_events = 10000
start_date = datetime.now() - timedelta(days=30)

doc_types = ['invoice', 'contract', 'receipt', 'tax_form']
languages = ['eng', 'spa', 'fra', 'deu']

user_events = pd.DataFrame({
    'user_id': np.random.randint(1, 1001, n_events),
    'event_timestamp': [start_date + timedelta(seconds=int(x)) 
                        for x in np.sort(np.random.randint(0, 30*24*3600, n_events))],
    'doc_type': np.random.choice(doc_types, n_events),
    'confidence_score': np.random.uniform(0.6, 1.0, n_events),
    'page_count': np.random.randint(1, 51, n_events),
    'has_tables': np.random.choice([True, False], n_events, p=[0.3, 0.7]),
    'language': np.random.choice(languages, n_events, p=[0.7, 0.15, 0.1, 0.05]),
    'processing_time_sec': np.random.lognormal(2.0, 0.8, n_events).astype(int)
})

user_events['created_at'] = datetime.now()

print(f"✓ Generated {len(user_events):,} user upload events")
print(f"  Date range: {user_events['event_timestamp'].min()} → {user_events['event_timestamp'].max()}")
print(f"  Unique users: {user_events['user_id'].nunique()}")

# Save locally first
local_path = "user_uploads.parquet"
user_events.to_parquet(local_path, index=False)
print(f"\n✓ Saved to local file: {local_path} ({os.path.getsize(local_path)/1024:.1f} KB)")

# Upload to ADLS
try:
    # Get storage account key from Azure ML workspace
    storage_account_url = f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net"
    
    # Use DefaultAzureCredential for ADLS access
    service_client = DataLakeServiceClient(
        account_url=storage_account_url,
        credential=credential
    )
    
    # Get or create container
    file_system_client = service_client.get_file_system_client(file_system=ADLS_CONTAINER)
    if not file_system_client.exists():
        file_system_client.create_file_system()
        print(f"✓ Created ADLS container: {ADLS_CONTAINER}")
    
    # Upload file
    today = datetime.now()
    file_path = f"user_uploads/year={today.year}/month={today.month:02d}/day={today.day:02d}/user_uploads.parquet"
    file_client = file_system_client.get_file_client(file_path)
    
    with open(local_path, "rb") as data:
        file_client.upload_data(data, overwrite=True)
    
    print(f"\n✓ Uploaded to ADLS: abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/{file_path}")
    print(f"  Storage cost: ~${os.path.getsize(local_path)/1024/1024 * 0.02:.4f}/month")
    
except Exception as e:
    print(f"\n❌ Error uploading to ADLS: {e}")
    print(f"  Continuing with local file for demonstration...")

## Cell 3: Define Azure ML Feature Set

Create feature set definitions using Azure ML SDK.

**Azure ML Feature Set** = Feast Feature View equivalent
- Source: ADLS Parquet files
- Entities: user_id
- Features: Aggregations over user activity

**Note:** Azure ML Feature Store uses declarative YAML + Python SDK (vs Feast's pure Python).

In [ ]:
from azure.ai.ml.entities import (
    FeatureStore,
    FeatureSet,
    FeatureSetSpecification,
    DataColumn,
    DataColumnType
)

# Define feature set specification
user_activity_spec = FeatureSetSpecification(
    source={
        "type": "parquet",
        "path": f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/user_uploads/"
    },
    entities=[
        DataColumn(name="user_id", type=DataColumnType.INTEGER)
    ],
    features=[
        DataColumn(name="avg_confidence_score", type=DataColumnType.FLOAT),
        DataColumn(name="total_pages_processed", type=DataColumnType.LONG),
        DataColumn(name="upload_count", type=DataColumnType.LONG),
        DataColumn(name="avg_processing_time_sec", type=DataColumnType.FLOAT),
    ],
    timestamp_column="event_timestamp"
)

# Create feature set
user_activity_feature_set = FeatureSet(
    name="user_activity_features",
    version="1",
    description="User activity aggregations over last 30 days",
    specification=user_activity_spec,
    tags={"team": "ml-infrastructure", "priority": "high"}
)

print("✓ Feature set definition created")
print(f"  Name: {user_activity_feature_set.name}")
print(f"  Version: {user_activity_feature_set.version}")
print(f"  Entities: {[e.name for e in user_activity_spec.entities]}")
print(f"  Features: {[f.name for f in user_activity_spec.features]}")
print(f"\n💡 This feature set will be registered to Azure ML workspace")
print(f"   and materialized to Azure Cache for Redis for online serving")

## Cell 4: Register Feature Set to Azure ML

Register the feature set with Azure ML Feature Store.

**What this does:**
1. Validates feature definitions
2. Registers metadata to Azure ML workspace
3. Creates lineage tracking (source → features)
4. Enables feature discovery in Azure ML Studio UI

**Portal URL:** https://ml.azure.com → Feature stores

In [ ]:
try:
    # Register feature set
    registered_feature_set = ml_client.feature_sets.begin_create_or_update(
        user_activity_feature_set
    ).result()
    
    print("✓ Feature set registered to Azure ML")
    print(f"  Resource ID: {registered_feature_set.id}")
    print(f"  Name: {registered_feature_set.name}")
    print(f"  Version: {registered_feature_set.version}")
    
    # Generate Azure ML Studio URL
    studio_url = (
        f"https://ml.azure.com/featureStore/detail/{WORKSPACE_NAME}/"
        f"featureSets/{registered_feature_set.name}/version/{registered_feature_set.version}"
        f"?wsid=/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
    print(f"\n🌐 View in Azure ML Studio:")
    print(f"  {studio_url}")
    
except Exception as e:
    print(f"❌ Error registering feature set: {e}")
    print(f"\n💡 Common issues:")
    print(f"  1. ADLS path not accessible (check storage account permissions)")
    print(f"  2. Feature store not enabled in workspace")
    print(f"  3. Invalid feature schema (check data types)")
    
    # For demonstration, we'll continue with local data
    print(f"\n⚠️  Continuing with local simulation...")

# List all registered feature sets
try:
    feature_sets = ml_client.feature_sets.list()
    print(f"\n📊 All feature sets in workspace:")
    for fs in feature_sets:
        print(f"  • {fs.name} (v{fs.version})")
except:
    print(f"\n📊 Feature sets: (unable to list, check permissions)")

## Cell 5: Materialize Features to Azure Redis

Compute aggregated features and write to Azure Cache for Redis (online store).

**Materialization job:**
1. Read user_uploads from ADLS
2. Compute aggregations (AVG, SUM, COUNT)
3. Write to Redis with TTL=30 days

**Redis key structure:**
```
fs:user_activity_features:user:12345:avg_confidence_score → 0.87
fs:user_activity_features:user:12345:total_pages_processed → 1250
```

**Cost:** Azure Cache for Redis Standard C1 (1GB) = $75/month

In [ ]:
import redis
import json

print("🔄 Materializing features to Azure Cache for Redis...\n")

# Compute aggregated features (same as main notebook)
user_activity = user_events.groupby('user_id').agg({
    'confidence_score': 'mean',
    'page_count': 'sum',
    'user_id': 'count',
    'processing_time_sec': 'mean'
}).rename(columns={
    'confidence_score': 'avg_confidence_score',
    'page_count': 'total_pages_processed',
    'user_id': 'upload_count',
    'processing_time_sec': 'avg_processing_time_sec'
}).reset_index()

print(f"✓ Computed features for {len(user_activity)} users")

# Connect to Azure Redis
try:
    r = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        password=REDIS_PASSWORD,
        ssl=True,
        ssl_cert_reqs=None
    )
    r.ping()
    print(f"✓ Connected to Azure Cache for Redis")
    
    # Materialize to Redis
    ttl_seconds = 30 * 24 * 3600  # 30 days
    feature_set_name = "user_activity_features"
    
    materialized_count = 0
    for _, row in user_activity.iterrows():
        user_id = int(row['user_id'])
        
        # Write each feature as separate key
        for feature_name in ['avg_confidence_score', 'total_pages_processed', 'upload_count', 'avg_processing_time_sec']:
            key = f"fs:{feature_set_name}:user:{user_id}:{feature_name}"
            value = float(row[feature_name])
            r.setex(key, ttl_seconds, value)
        
        materialized_count += 1
        
        if materialized_count % 100 == 0:
            print(f"  Materialized {materialized_count}/{len(user_activity)} users...", end="\r")
    
    print(f"\n✓ Materialized {materialized_count} users to Redis                      ")
    print(f"  Total keys: {materialized_count * 4:,}")
    print(f"  TTL: 30 days")
    
    # Verify materialization
    test_user_id = 42
    test_key = f"fs:{feature_set_name}:user:{test_user_id}:avg_confidence_score"
    test_value = r.get(test_key)
    if test_value:
        print(f"\n✓ Verification: user {test_user_id} avg_confidence_score = {float(test_value):.3f}")
    
    # Redis memory usage
    info = r.info('memory')
    used_memory_mb = info['used_memory'] / 1024 / 1024
    print(f"\n📊 Redis memory usage: {used_memory_mb:.2f} MB")
    print(f"   Cost estimate: ~$75/month (Standard C1, 1GB)")
    
except Exception as e:
    print(f"❌ Error materializing to Redis: {e}")
    print(f"   Check REDIS_HOST and REDIS_PASSWORD configuration")

## Cell 6: Online Serving with Azure Functions

Deploy serverless feature serving API using Azure Functions.

**Architecture:**
```
Client → Azure Functions (HTTP trigger)
           ↓
        Azure Cache for Redis (feature lookup)
           ↓
        Return features (JSON)
```

**Cost:** $0.20/million requests + $0.000016/GB-s execution time

**Deployment:** Via Azure Functions Core Tools or Azure Portal

In [ ]:
# Azure Function code (for reference — deploy separately)
azure_function_code = '''
import azure.functions as func
import redis
import json
import os

# Initialize Redis connection (reused across invocations)
r = redis.Redis(
    host=os.environ["REDIS_HOST"],
    port=int(os.environ["REDIS_PORT"]),
    password=os.environ["REDIS_PASSWORD"],
    ssl=True,
    ssl_cert_reqs=None
)

def main(req: func.HttpRequest) -> func.HttpResponse:
    """Fetch features for a user from Redis."""
    
    # Parse request
    user_id = req.params.get('user_id')
    if not user_id:
        return func.HttpResponse(
            json.dumps({"error": "Missing user_id parameter"}),
            status_code=400,
            mimetype="application/json"
        )
    
    # Fetch features from Redis
    feature_set = "user_activity_features"
    features = {}
    
    for feature_name in ['avg_confidence_score', 'total_pages_processed', 'upload_count']:
        key = f"fs:{feature_set}:user:{user_id}:{feature_name}"
        value = r.get(key)
        features[feature_name] = float(value) if value else None
    
    # Return JSON response
    return func.HttpResponse(
        json.dumps({
            "user_id": user_id,
            "features": features
        }),
        status_code=200,
        mimetype="application/json"
    )
'''

print("📦 Azure Function: Feature Serving API\n")
print("📄 Function code (deploy to Azure Functions):")
print(azure_function_code)

# Simulate local invocation
print("\n🧪 Simulating Azure Function call...\n")

import time

def simulate_azure_function(user_id):
    """Simulate Azure Function invocation."""
    try:
        r = redis.Redis(
            host=REDIS_HOST,
            port=REDIS_PORT,
            password=REDIS_PASSWORD,
            ssl=True,
            ssl_cert_reqs=None
        )
        
        feature_set = "user_activity_features"
        features = {}
        
        for feature_name in ['avg_confidence_score', 'total_pages_processed', 'upload_count']:
            key = f"fs:{feature_set}:user:{user_id}:{feature_name}"
            value = r.get(key)
            features[feature_name] = float(value) if value else None
        
        return {
            "user_id": user_id,
            "features": features,
            "status": 200
        }
    except Exception as e:
        return {"error": str(e), "status": 500}

# Test API call
test_user_id = 42
start = time.perf_counter()
response = simulate_azure_function(test_user_id)
latency_ms = (time.perf_counter() - start) * 1000

print(f"GET /api/features?user_id={test_user_id}")
print(f"\n📊 Response (HTTP {response.get('status', 200)}):")
print(json.dumps(response, indent=2))
print(f"\n⚡ Latency: {latency_ms:.2f} ms")
print(f"   (includes Redis lookup + JSON serialization)")

# Cost calculation
print(f"\n💰 Cost estimate (10k requests/day):")
monthly_requests = 10000 * 30
execution_time_sec = 0.05  # 50ms average
memory_gb = 0.128  # 128MB default

invocation_cost = (monthly_requests / 1_000_000) * 0.20
execution_cost = (monthly_requests * execution_time_sec * memory_gb) * 0.000016
total_cost = invocation_cost + execution_cost

print(f"  Invocations: ${invocation_cost:.4f}/month ({monthly_requests:,} requests)")
print(f"  Execution: ${execution_cost:.4f}/month ({execution_time_sec}s/request, {memory_gb}GB)")
print(f"  Total: ${total_cost:.2f}/month")
print(f"\n  vs Redis cost: $75/month (Standard C1)")
print(f"  Combined: ${total_cost + 75:.2f}/month")

## Cell 7: Cost Analysis — Redis Tiers

Compare Azure Cache for Redis pricing tiers.

**Tiers:**
- **Basic**: No SLA, single node (dev/test only)
- **Standard**: 99.9% SLA, replication, no persistence
- **Premium**: 99.95% SLA, persistence, clustering, VNet
- **Enterprise**: 99.99% SLA, Redis modules (RediSearch, RedisJSON)

**Recommendation:** Standard C1 (1GB) for <10k users, Premium P1 (6GB) for production

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Azure Cache for Redis pricing (as of 2024)
redis_tiers = [
    {"name": "Basic C0", "memory_gb": 0.25, "cost_month": 16, "sla": "None", "use_case": "Dev/test"},
    {"name": "Basic C1", "memory_gb": 1, "cost_month": 55, "sla": "None", "use_case": "Dev/test"},
    {"name": "Standard C1", "memory_gb": 1, "cost_month": 75, "sla": "99.9%", "use_case": "Production (<10k users)"},
    {"name": "Standard C2", "memory_gb": 2.5, "cost_month": 155, "sla": "99.9%", "use_case": "Production (<25k users)"},
    {"name": "Premium P1", "memory_gb": 6, "cost_month": 358, "sla": "99.95%", "use_case": "Production (50k users)"},
    {"name": "Premium P2", "memory_gb": 13, "cost_month": 716, "sla": "99.95%", "use_case": "Production (100k users)"},
    {"name": "Enterprise E10", "memory_gb": 12, "cost_month": 1820, "sla": "99.99%", "use_case": "Enterprise (modules)"},
]

print("💰 Azure Cache for Redis — Pricing Tiers\n")
print(f"{'Tier':<18} {'Memory':<10} {'Cost/Month':<12} {'SLA':<8} {'Use Case'}")
print("=" * 80)
for tier in redis_tiers:
    print(f"{tier['name']:<18} {tier['memory_gb']:<10}GB ${tier['cost_month']:<11} {tier['sla']:<8} {tier['use_case']}")

# Plot cost vs memory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#1a1a2e')

# Cost comparison
names = [t['name'] for t in redis_tiers]
costs = [t['cost_month'] for t in redis_tiers]
colors = ['red' if 'Basic' in n else 'orange' if 'Standard' in n else 'green' if 'Premium' in n else 'blue' for n in names]

ax1.barh(names, costs, color=colors, alpha=0.7)
ax1.set_xlabel('Cost ($/month)', color='white')
ax1.set_title('Azure Redis Pricing', color='white')
ax1.tick_params(colors='white')
ax1.set_facecolor('#1a1a2e')
ax1.spines['bottom'].set_color('white')
ax1.spines['left'].set_color('white')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='x', alpha=0.3, color='white')

# Cost per GB
cost_per_gb = [t['cost_month'] / t['memory_gb'] for t in redis_tiers]
ax2.barh(names, cost_per_gb, color=colors, alpha=0.7)
ax2.set_xlabel('Cost per GB ($/month)', color='white')
ax2.set_title('Cost Efficiency', color='white')
ax2.tick_params(colors='white')
ax2.set_facecolor('#1a1a2e')
ax2.spines['bottom'].set_color('white')
ax2.spines['left'].set_color('white')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='x', alpha=0.3, color='white')

plt.tight_layout()
plt.savefig('../img/ch08-azure-redis-pricing.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

# Recommendation
print("\n📊 Recommendation for InferenceBase:")
print("  Current: 1,000 active users (10k requests/day)")
print("  Feature size: ~500 bytes per user (8 features × ~60 bytes)")
print("  Total memory: 1,000 users × 500 bytes = 0.5 MB (negligible)")
print("\n  ✅ Start with: Standard C1 (1GB, $75/month)")
print("  ✅ Scale to: Standard C2 (2.5GB, $155/month) at 10k users")
print("  ✅ Upgrade to: Premium P1 (6GB, $358/month) when need 99.95% SLA")
print("\n💡 Cost vs PostgreSQL:")
print("  Before: 4 PostgreSQL replicas = $800/month")
print("  After: 1 Redis Standard C1 = $75/month")
print("  Savings: $725/month (91% reduction)")

## Cell 8: Monitor Feature Serving with Application Insights

Track feature serving metrics using Azure Application Insights.

**Metrics to monitor:**
1. **Latency**: p50, p95, p99 of feature lookup time
2. **Cache hit rate**: % of requests served from Redis (vs fallback to DB)
3. **Feature staleness**: Time since last materialization
4. **Error rate**: % of failed feature lookups

**Alerts:**
- p95 latency > 20ms → scale Redis tier
- Cache hit rate < 95% → increase TTL or materialization frequency
- Feature staleness > 2 hours → materialization job failed

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Simulate feature serving metrics over 24 hours
np.random.seed(42)
hours = 24
timestamps = [datetime.now() - timedelta(hours=hours-i) for i in range(hours)]

# Latency metrics (ms)
p50_latency = np.random.normal(6, 1, hours)  # Redis optimized
p95_latency = np.random.normal(12, 2, hours)
p99_latency = np.random.normal(18, 3, hours)

# Cache hit rate (%)
cache_hit_rate = np.random.normal(98, 1, hours)
cache_hit_rate = np.clip(cache_hit_rate, 95, 100)

# Feature staleness (minutes since last materialization)
staleness_minutes = []
last_materialization = 0
for i in range(hours):
    if i % 1 == 0:  # Materialize every hour
        last_materialization = 0
    else:
        last_materialization += 60  # Minutes
    staleness_minutes.append(last_materialization)

# Error rate (%)
error_rate = np.random.exponential(0.1, hours)  # Low error rate

print("📊 Feature Serving Metrics (Last 24 Hours)\n")
print(f"{'Metric':<25} {'Mean':<12} {'Min':<12} {'Max':<12}")
print("=" * 65)
print(f"{'p50 Latency (ms)':<25} {np.mean(p50_latency):<12.2f} {np.min(p50_latency):<12.2f} {np.max(p50_latency):<12.2f}")
print(f"{'p95 Latency (ms)':<25} {np.mean(p95_latency):<12.2f} {np.min(p95_latency):<12.2f} {np.max(p95_latency):<12.2f}")
print(f"{'p99 Latency (ms)':<25} {np.mean(p99_latency):<12.2f} {np.min(p99_latency):<12.2f} {np.max(p99_latency):<12.2f}")
print(f"{'Cache Hit Rate (%)':<25} {np.mean(cache_hit_rate):<12.2f} {np.min(cache_hit_rate):<12.2f} {np.max(cache_hit_rate):<12.2f}")
print(f"{'Feature Staleness (min)':<25} {np.mean(staleness_minutes):<12.2f} {np.min(staleness_minutes):<12.2f} {np.max(staleness_minutes):<12.2f}")
print(f"{'Error Rate (%)':<25} {np.mean(error_rate):<12.2f} {np.min(error_rate):<12.2f} {np.max(error_rate):<12.2f}")

# Plot monitoring dashboard
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10), facecolor='#1a1a2e')

# Latency
ax1.plot(timestamps, p50_latency, label='p50', color='green', linewidth=2)
ax1.plot(timestamps, p95_latency, label='p95', color='orange', linewidth=2)
ax1.plot(timestamps, p99_latency, label='p99', color='red', linewidth=2)
ax1.axhline(y=10, color='yellow', linestyle='--', alpha=0.5, label='10ms target')
ax1.set_xlabel('Time', color='white')
ax1.set_ylabel('Latency (ms)', color='white')
ax1.set_title('Feature Lookup Latency', color='white', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax1.tick_params(colors='white')
ax1.set_facecolor('#1a1a2e')
ax1.grid(alpha=0.3, color='white')
for spine in ax1.spines.values():
    spine.set_color('white')

# Cache hit rate
ax2.plot(timestamps, cache_hit_rate, color='green', linewidth=2)
ax2.axhline(y=95, color='red', linestyle='--', alpha=0.5, label='95% threshold')
ax2.set_xlabel('Time', color='white')
ax2.set_ylabel('Cache Hit Rate (%)', color='white')
ax2.set_title('Redis Cache Hit Rate', color='white', fontsize=14, fontweight='bold')
ax2.set_ylim([90, 100])
ax2.legend(loc='lower right', facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax2.tick_params(colors='white')
ax2.set_facecolor('#1a1a2e')
ax2.grid(alpha=0.3, color='white')
for spine in ax2.spines.values():
    spine.set_color('white')

# Feature staleness
ax3.plot(timestamps, staleness_minutes, color='orange', linewidth=2)
ax3.axhline(y=60, color='green', linestyle='--', alpha=0.5, label='Materialization interval')
ax3.set_xlabel('Time', color='white')
ax3.set_ylabel('Staleness (minutes)', color='white')
ax3.set_title('Feature Staleness', color='white', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax3.tick_params(colors='white')
ax3.set_facecolor('#1a1a2e')
ax3.grid(alpha=0.3, color='white')
for spine in ax3.spines.values():
    spine.set_color('white')

# Error rate
ax4.plot(timestamps, error_rate, color='red', linewidth=2)
ax4.axhline(y=0.5, color='yellow', linestyle='--', alpha=0.5, label='0.5% alert threshold')
ax4.set_xlabel('Time', color='white')
ax4.set_ylabel('Error Rate (%)', color='white')
ax4.set_title('Feature Lookup Errors', color='white', fontsize=14, fontweight='bold')
ax4.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax4.tick_params(colors='white')
ax4.set_facecolor('#1a1a2e')
ax4.grid(alpha=0.3, color='white')
for spine in ax4.spines.values():
    spine.set_color('white')

plt.suptitle('Feature Store Monitoring Dashboard', color='white', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../img/ch08-azure-monitoring.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n✅ Monitoring dashboard saved: ../img/ch08-azure-monitoring.png")

# Alert recommendations
print("\n🔔 Recommended Application Insights Alerts:\n")
alerts = [
    {"metric": "p95 Latency", "threshold": ">20ms for 5 min", "action": "Scale Redis tier (C1→C2)"},
    {"metric": "Cache Hit Rate", "threshold": "<95% for 10 min", "action": "Check materialization job, increase TTL"},
    {"metric": "Feature Staleness", "threshold": ">120 min", "action": "Materialization job failed, trigger manual run"},
    {"metric": "Error Rate", "threshold": ">1% for 5 min", "action": "Redis connection issue, check firewall/VNet"},
]

for alert in alerts:
    print(f"  • {alert['metric']:<20} {alert['threshold']:<25} → {alert['action']}")

print("\n✅ Chapter 8 Azure supplement complete!")
print("\n🎯 Key achievements:")
print("  • Production Redis: Azure Cache Standard C1 ($75/month)")
print("  • Serverless serving: Azure Functions ($0.20/M requests)")
print("  • Monitoring: Application Insights (latency, cache hit rate, staleness)")
print("  • Total cost: ~$75-150/month (vs $800/month PostgreSQL replicas)")
print("\n➡️  Next: Ch.9 — ML Experiment Tracking (MLflow + Azure ML)")